In [114]:
import pandas as pd
import numpy as np

df = pd.read_csv("UCI_Credit_Card.csv")
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [ ]:
# fixing all the columns as of now
# Fix EDUCATION (5 and 6 are officially "unknown", 0 is undocumented)
edu_map = {
    1: "Graduate",
    2: "University",
    3: "HighSchool",
    4: "Other",
    5: "Other",
    6: "Other",
    0: "Other",
}
# df["EDUCATION"] = df["EDUCATION"].map(edu_map)

# Verify
# print(df["EDUCATION"].value_counts())
# print("NaNs:", df["EDUCATION"].isnull().sum())


marital_map = {
    1: "Married",
    2: "Single",
    3: "Other",
    0: "Other", 
}
# df["MARRIAGE"] = df["MARRIAGE"].map(marital_map)


sex_map = {
    1: "Male", 
    2: "Female"
    }
# df["SEX"] = df["SEX"].map(sex_map)

# Verify
# print(df["SEX"].value_counts())
# print("NaNs:", df["SEX"].isnull().sum())

# Dropping the ID and Renaming the last column
# df.drop('ID', axis=1, inplace=True)

df.rename(columns={"default.payment.next.month":"Default"},inplace=True)

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,Default
0,20000.0,Female,University,Married,24,2,2,-1,-1,-2,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,120000.0,Female,University,Single,26,-1,2,0,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,90000.0,Female,University,Single,34,0,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,50000.0,Female,University,Married,37,0,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,50000.0,Male,University,Married,57,-1,0,-1,0,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000.0,Male,HighSchool,Married,39,0,0,0,0,0,...,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0,0
29996,150000.0,Male,HighSchool,Single,43,-1,-1,-1,-1,0,...,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0,0
29997,30000.0,Male,University,Single,37,4,3,2,-1,0,...,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0,1
29998,80000.0,Male,HighSchool,Married,41,1,-1,0,0,0,...,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0,1


In [129]:
# Utilization
df["utilization"] = df["BILL_AMT1"] / (df["LIMIT_BAL"] + 1)
# print(df["utilization"].describe().round(3))


# Payment Ratio Calculation
bill_cols = [
    "BILL_AMT1",
    "BILL_AMT2",
    "BILL_AMT3",
    "BILL_AMT4",
    "BILL_AMT5",
    "BILL_AMT6",
]
pay_cols = [
    "PAY_AMT1", 
    "PAY_AMT2", 
    "PAY_AMT3", 
    "PAY_AMT4", 
    "PAY_AMT5", 
    "PAY_AMT6"
    ]

bill_total = df[bill_cols].sum(axis=1).replace(0, np.nan)
pay_total = df[pay_cols].sum(axis=1)

df["payment_ratio"] = (pay_total / bill_total).clip(0, 1)


pay_status_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
df["late_months"] = (df[pay_status_cols] > 0).sum(axis=1)

print(df["late_months"].value_counts().sort_index())


df["bill_trend"] = df["BILL_AMT1"] - df["BILL_AMT6"]
df

late_months
0    19931
1     4426
2     1899
3     1154
4      951
5      298
6     1341
Name: count, dtype: int64


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,Default,utilization,payment_ratio,late_months,bill_trend
0,20000.0,Female,University,Married,24,2,2,-1,-1,-2,...,689.0,0.0,0.0,0.0,0.0,1,0.195640,0.089434,2,3913.0
1,120000.0,Female,University,Single,26,-1,2,0,0,0,...,1000.0,1000.0,1000.0,0.0,2000.0,1,0.022350,0.292791,2,-579.0
2,90000.0,Female,University,Single,34,0,0,0,0,0,...,1500.0,1000.0,1000.0,1000.0,5000.0,0,0.324874,0.108388,0,13690.0
3,50000.0,Female,University,Married,37,0,0,0,0,0,...,2019.0,1200.0,1100.0,1069.0,1000.0,0,0.939781,0.036259,0,17443.0
4,50000.0,Male,University,Married,57,-1,0,-1,0,0,...,36681.0,10000.0,9000.0,689.0,679.0,0,0.172337,0.540054,0,-10514.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000.0,Male,HighSchool,Married,39,0,0,0,0,0,...,20000.0,5003.0,3047.0,5000.0,1000.0,0,0.858851,0.058661,0,172968.0
29996,150000.0,Male,HighSchool,Single,43,-1,-1,-1,-1,0,...,3526.0,8998.0,129.0,0.0,0.0,0,0.011220,0.684071,0,1683.0
29997,30000.0,Male,University,Single,37,4,3,2,-1,0,...,0.0,22000.0,4200.0,2000.0,3100.0,1,0.118829,0.443997,3,-15792.0
29998,80000.0,Male,HighSchool,Married,41,1,-1,0,0,0,...,3409.0,1178.0,1926.0,52964.0,1804.0,1,-0.020562,0.552044,1,-50589.0
